# Model Building for Phishing URL Detection

## 1. Introduction

Machine learning models are used in this project to classify websites as phishing or legitimate based on URL and webpage-related characteristics.

The previous notebooks completed dataset description, data preprocessing, exploratory data analysis, feature engineering, and feature selection.

In this notebook, the selected predictive features will be used to train multiple machine learning classification models.

## 2. Objectives

The main objectives of this notebook are:

- Load the selected feature dataset.
- Separate predictive features from the target variable.
- Create a leakage-free training and testing dataset.
- Apply feature scaling where required.
- Train multiple classification models.
- Record model configurations and training results.
- Generate predictions from the trained models.
- Calculate baseline classification metrics.
- Perform cross-validation on the training data.
- Prepare the models for the subsequent hyperparameter-tuning stage.

## 3. Classification Task

This project is a binary classification problem.

The target variable is:

- `0` = Phishing
- `1` = Legitimate

The machine learning models will learn patterns in the selected URL and webpage features to distinguish between these two classes.

## 4. Models

The following classification algorithms will be trained:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Support Vector Machine

These models represent different machine learning approaches and allow their baseline performance to be compared.

## 5. Train-Test Strategy

The URL-group splitting strategy established during preprocessing will be maintained.

The same URL must not occur in both the training and testing datasets.

This prevents repeated URLs from introducing information leakage and provides a more reliable evaluation of model performance.

## 6. Feature Scaling

Feature scaling will be applied where required.

- Logistic Regression requires scaled numerical features for stable optimization.
- Support Vector Machine is sensitive to feature scale and therefore requires scaling.
- Decision Tree and Random Forest are tree-based models and do not require feature scaling.

The scaler will be fitted using training data only and then applied to the test data.

## 7. Model Evaluation

The baseline models will be evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

Cross-validation will also be performed using the training dataset to examine model consistency.

## 8. Expected Outcome

At the end of this notebook, multiple baseline classification models will have been trained and evaluated.

Their results will establish a baseline for the subsequent hyperparameter-tuning and final model-evaluation stages.

In [1]:
# Import libraries and load the selected feature dataset

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split

dataset_path = "../dataset/PhiUSIIL_Selected_Features.csv"

df = pd.read_csv(dataset_path)

print("MODEL BUILDING ENVIRONMENT READY")
print("=" * 60)
print("Dataset shape:", df.shape)
print("Number of records:", len(df))
print("Number of columns:", len(df.columns))

MODEL BUILDING ENVIRONMENT READY
Dataset shape: (235795, 41)
Number of records: 235795
Number of columns: 41


In [2]:
# Separate predictive features (X) and target variable (y)

target_column = "label"

# X contains all selected predictive features
X = df.drop(columns=[target_column])

# y contains the target variable
y = df[target_column]

print("FEATURE AND TARGET PREPARATION")
print("=" * 60)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nNumber of input features:", X.shape[1])

print("\nTarget classes:")
print(y.value_counts().sort_index())

print("\nTarget mapping:")
print("0 = Phishing")
print("1 = Legitimate")

FEATURE AND TARGET PREPARATION
X shape: (235795, 40)
y shape: (235795,)

Number of input features: 40

Target classes:
label
0    100945
1    134850
Name: count, dtype: int64

Target mapping:
0 = Phishing
1 = Legitimate


In [5]:
# Create a leakage-free URL-group train/test split

print("URL-GROUP TRAIN-TEST SPLIT")
print("=" * 60)

# Load URL information only once
url_data = pd.read_csv(
    "../dataset/PhiUSIIL_Engineered_Features.csv",
    usecols=["URL", "label"]
)

# Create unique URL groups
url_groups = url_data.drop_duplicates(subset="URL")

# Split unique URLs
train_url_groups, test_url_groups = train_test_split(
    url_groups,
    test_size=0.20,
    random_state=42,
    stratify=url_groups["label"]
)

# Create URL sets
train_url_set = set(train_url_groups["URL"])
test_url_set = set(test_url_groups["URL"])

# Match URLs directly using the original engineered dataset
train_mask = url_data["URL"].isin(train_url_set)
test_mask = url_data["URL"].isin(test_url_set)

# Create train/test datasets
X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining percentage:",
      round(len(X_train) / len(df) * 100, 2), "%")

print("Testing percentage:",
      round(len(X_test) / len(df) * 100, 2), "%")

# Verify URL leakage
common_urls = train_url_set.intersection(test_url_set)

print("\nURLs appearing in both train and test:", len(common_urls))

if len(common_urls) == 0:
    print("✅ No URL leakage detected.")
else:
    print("❌ URL leakage detected.")

URL-GROUP TRAIN-TEST SPLIT
Training records: 188644
Testing records: 47151

Training percentage: 80.0 %
Testing percentage: 20.0 %

URLs appearing in both train and test: 0
✅ No URL leakage detected.


In [6]:
# Feature scaling using training data only

from sklearn.preprocessing import StandardScaler

print("FEATURE SCALING")
print("=" * 60)

# Create scaler
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the same scaler
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)

print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)

print("\nTraining mean after scaling:",
      round(X_train_scaled.mean().mean(), 4))

print("Training standard deviation:",
      round(X_train_scaled.std().mean(), 4))

print("\n✅ Scaling completed.")
print("Scaler was fitted only on training data.")

FEATURE SCALING
Training data shape: (188644, 40)
Testing data shape: (47151, 40)

Training mean after scaling: 0.0
Training standard deviation: 1.0

✅ Scaling completed.
Scaler was fitted only on training data.


In [7]:
# Define baseline machine learning models

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

print("BASELINE MODEL CONFIGURATION")
print("=" * 60)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Support Vector Machine": SVC(
        probability=True,
        random_state=42
    )
}

print("Models configured:")
for i, model_name in enumerate(models.keys(), start=1):
    print(f"{i}. {model_name}")

print("\nTotal baseline models:", len(models))

BASELINE MODEL CONFIGURATION
Models configured:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Support Vector Machine

Total baseline models: 4


In [8]:
# Train baseline machine learning models

import time

print("BASELINE MODEL TRAINING")
print("=" * 60)

trained_models = {}
training_times = {}

# Models suitable for the full training dataset
models_to_train = {
    "Logistic Regression": models["Logistic Regression"],
    "Decision Tree": models["Decision Tree"],
    "Random Forest": models["Random Forest"]
}

for model_name, model in models_to_train.items():

    print(f"\nTraining {model_name}...")

    start_time = time.time()

    model.fit(X_train_scaled, y_train)

    end_time = time.time()

    elapsed_time = end_time - start_time

    trained_models[model_name] = model
    training_times[model_name] = elapsed_time

    print(f"✅ {model_name} trained successfully.")
    print(f"Training time: {elapsed_time:.2f} seconds")

print("\n" + "=" * 60)
print("BASELINE TRAINING COMPLETED")

print("\nTraining times:")
for model_name, elapsed_time in training_times.items():
    print(f"{model_name}: {elapsed_time:.2f} seconds")

BASELINE MODEL TRAINING

Training Logistic Regression...
✅ Logistic Regression trained successfully.
Training time: 0.42 seconds

Training Decision Tree...
✅ Decision Tree trained successfully.
Training time: 1.39 seconds

Training Random Forest...
✅ Random Forest trained successfully.
Training time: 8.40 seconds

BASELINE TRAINING COMPLETED

Training times:
Logistic Regression: 0.42 seconds
Decision Tree: 1.39 seconds
Random Forest: 8.40 seconds


In [9]:
# Generate predictions from the trained baseline models

print("GENERATING MODEL PREDICTIONS")
print("=" * 60)

predictions = {}
probabilities = {}

for model_name, model in trained_models.items():

    print(f"\nGenerating predictions for {model_name}...")

    # Class predictions
    y_pred = model.predict(X_test_scaled)

    # Probability of class 1 (Legitimate)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    predictions[model_name] = y_pred
    probabilities[model_name] = y_prob

    print(f"✅ Predictions generated.")
    print("Number of predictions:", len(y_pred))

print("\n" + "=" * 60)
print("Prediction generation completed.")

GENERATING MODEL PREDICTIONS

Generating predictions for Logistic Regression...
✅ Predictions generated.
Number of predictions: 47151

Generating predictions for Decision Tree...
✅ Predictions generated.
Number of predictions: 47151

Generating predictions for Random Forest...
✅ Predictions generated.
Number of predictions: 47151

Prediction generation completed.


In [10]:
# Calculate baseline evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("BASELINE MODEL EVALUATION")
print("=" * 60)

baseline_results = []

for model_name in trained_models.keys():

    y_pred = predictions[model_name]
    y_prob = probabilities[model_name]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    baseline_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": roc_auc
    })

baseline_results_df = pd.DataFrame(baseline_results)

display(
    baseline_results_df.round(4)
)

print("\n✅ Baseline evaluation completed.")

BASELINE MODEL EVALUATION


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.9999,0.9999,1.0,0.9999,1.0
1,Decision Tree,1.0000,1.0000,1.0,1.0000,1.0
2,Random Forest,1.0000,1.0000,1.0,1.0000,1.0



✅ Baseline evaluation completed.


In [11]:
# Perform 5-fold cross-validation on the training data

from sklearn.model_selection import StratifiedKFold, cross_validate

print("5-FOLD CROSS-VALIDATION")
print("=" * 60)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for model_name, model in trained_models.items():

    print(f"\nRunning cross-validation for {model_name}...")

    scores = cross_validate(
        model,
        X_train_scaled,
        y_train,
        cv=cv,
        scoring=[
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc"
        ],
        n_jobs=-1
    )

    cv_results.append({
        "Model": model_name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1-Score": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_results_df = pd.DataFrame(cv_results)

print("\n" + "=" * 60)
print("CROSS-VALIDATION RESULTS")

display(cv_results_df.round(4))

5-FOLD CROSS-VALIDATION

Running cross-validation for Logistic Regression...

Running cross-validation for Decision Tree...

Running cross-validation for Random Forest...

CROSS-VALIDATION RESULTS


,Model,CV Accuracy,CV Precision,CV Recall,CV F1-Score,CV ROC-AUC
0,Logistic Regression,0.9998,0.9997,1.0,0.9999,1.0
1,Decision Tree,1.0000,1.0000,1.0,1.0000,1.0
2,Random Forest,1.0000,1.0000,1.0,1.0000,1.0


In [12]:
# Combine test-set and cross-validation results

print("MODEL RESULTS SUMMARY")
print("=" * 60)

# Rename CV columns for merging
cv_summary = cv_results_df.copy()

# Merge baseline test results with cross-validation results
model_results = pd.merge(
    baseline_results_df,
    cv_summary,
    on="Model",
    how="left"
)

# Add training time
model_results["Training Time (sec)"] = (
    model_results["Model"].map(training_times)
)

# Arrange columns
model_results = model_results[
    [
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC",
        "CV Accuracy",
        "CV Precision",
        "CV Recall",
        "CV F1-Score",
        "CV ROC-AUC",
        "Training Time (sec)"
    ]
]

display(model_results.round(4))

print("\nNumber of baseline models evaluated:",
      len(model_results))

print("\n✅ Model results summary created.")

MODEL RESULTS SUMMARY


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,CV Accuracy,CV Precision,CV Recall,CV F1-Score,CV ROC-AUC,Training Time (sec)
0,Logistic Regression,0.9999,0.9999,1.0,0.9999,1.0,0.9998,0.9997,1.0,0.9999,1.0,0.4228
1,Decision Tree,1.0000,1.0000,1.0,1.0000,1.0,1.0000,1.0000,1.0,1.0000,1.0,1.3852
2,Random Forest,1.0000,1.0000,1.0,1.0000,1.0,1.0000,1.0000,1.0,1.0000,1.0,8.3998



Number of baseline models evaluated: 3

✅ Model results summary created.


In [13]:
# Save baseline model results and trained models

import joblib

print("SAVING MODEL BUILDING RESULTS")
print("=" * 60)

# Create required directories
os.makedirs("../models", exist_ok=True)
os.makedirs("../results", exist_ok=True)

# Save model results
results_path = "../results/baseline_model_results.csv"

model_results.to_csv(
    results_path,
    index=False
)

# Save each trained baseline model
for model_name, model in trained_models.items():

    file_name = (
        model_name.lower()
        .replace(" ", "_")
        + "_baseline.pkl"
    )

    model_path = os.path.join(
        "../models",
        file_name
    )

    joblib.dump(model, model_path)

    print(f"✅ Saved: {model_path}")

# Save scaler
scaler_path = "../models/feature_scaler.pkl"

joblib.dump(
    scaler,
    scaler_path
)

print(f"✅ Saved: {scaler_path}")

print("\n" + "=" * 60)
print("MODEL BUILDING COMPLETED")
print("Results saved to:", results_path)

SAVING MODEL BUILDING RESULTS
✅ Saved: ../models\logistic_regression_baseline.pkl
✅ Saved: ../models\decision_tree_baseline.pkl
✅ Saved: ../models\random_forest_baseline.pkl
✅ Saved: ../models/feature_scaler.pkl

MODEL BUILDING COMPLETED
Results saved to: ../results/baseline_model_results.csv
